In [ ]:
# ===== NOTEBOOK A — CELL 0R : RESUME FROM UPLOADED CACHE =====
# essentials + img384 দুটো Kaggle Dataset হিসেবে attach করা আছে ধরে নিচ্ছি।
import os, glob
COMP   = '/kaggle/input/competitions/astroclimb'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']
OUT    = '/kaggle/working'          # নতুন যা বানাবে তা এখানেই যাবে

# dataset-এর slug/folder যাই হোক, নিজেই খুঁজে নেয় — path hardcode করার দরকার নেই
W     = os.path.dirname(glob.glob('/kaggle/input/**/emb_clip_img.npy', recursive=True)[0])
IMGD  = os.path.dirname(glob.glob('/kaggle/input/**/*.jpg', recursive=True)[0])  # zip root-এ খুললেও চলবে
CACHE = W                            # cell 9 এই নামটাই চায়
print('W    =', W)
print('IMGD =', IMGD, '| jpgs:', len(os.listdir(IMGD)), '(expect 14598)')

need = ['meta_train.parquet','meta_test.parquet','texts.parquet',
        'img_hashes.parquet','txt_hashes.parquet',
        'emb_clip_img.npy','emb_clip_txt.npy','emb_sci_txt.npy','emb_dino_img.npy']
miss = [f for f in need if not os.path.exists(f'{W}/{f}')]
print('\nMISSING:', miss if miss else 'কিছু না ✅ — সোজা cell 15 (build) থেকে চালাও')


In [1]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): 10 GB CSV পুরোটা RAM-এ — Stage 0, একবারই দরকার ছিল
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
import os, pandas as pd, numpy as np

COMP = '/kaggle/input/competitions/astroclimb'
print(os.listdir(COMP))
for f in os.listdir(COMP):
    print(f, round(os.path.getsize(f'{COMP}/{f}')/1e9, 2), 'GB')

tr = pd.read_csv(f'{COMP}/train.csv')
te = pd.read_csv(f'{COMP}/test.csv')

print('\ntrain', tr.shape, '| test', te.shape)
print('train cols:', tr.columns.tolist())
print('test  cols:', te.columns.tolist())
print('\nnulls train:\n', tr.isna().sum())

LABELS = ['same_figure', 'same_paper', 'related_papers', 'unrelated_papers']
rs = tr[LABELS].sum(axis=1)
print('\nexactly one label:', int((rs == 1).sum()), '/', len(tr),
      '| zero:', int((rs == 0).sum()), '| multi:', int((rs > 1).sum()))
print('label cols leaked into test (should be []):',
      [c for c in LABELS if c in te.columns])


['sample_submission.csv', 'train.csv', 'test.csv', 'solution.csv']
sample_submission.csv 0.0 GB
train.csv 10.1 GB
test.csv 10.13 GB
solution.csv 0.0 GB

train (10000, 7) | test (10000, 3)
train cols: ['id', 'same_figure', 'same_paper', 'related_papers', 'unrelated_papers', 'obj_1', 'obj_2']
test  cols: ['id', 'obj_1', 'obj_2']

nulls train:
 id                  0
same_figure         0
same_paper          0
related_papers      0
unrelated_papers    0
obj_1               0
obj_2               0
dtype: int64

exactly one label: 10000 / 10000 | zero: 0 | multi: 0
label cols leaked into test (should be []): []


In [2]:
import pandas as pd
COMP = '/kaggle/input/competitions/astroclimb'

ss = pd.read_csv(f'{COMP}/sample_submission.csv')
print('sample_submission', ss.shape, ss.columns.tolist()); print(ss.head(3))

sol = pd.read_csv(f'{COMP}/solution.csv')
print('\nsolution', sol.shape, sol.columns.tolist()); print(sol.head(3))
print('\nsolution label sums:\n', sol.select_dtypes('number').sum())

# বড় CSV থেকে শুধু ৩ row — obj column গুলো truncate করে দেখাচ্ছি
head = pd.read_csv(f'{COMP}/train.csv', nrows=3)
print('\ntrain cols:', head.columns.tolist())
print(head.assign(obj_1=head.obj_1.str[:40], obj_2=head.obj_2.str[:40]).to_string())


sample_submission (10000, 6) ['id', 'same_figure', 'same_paper', 'related_papers', 'unrelated_papers', 'Usage']
   id  same_figure  same_paper  related_papers  unrelated_papers   Usage
0   0            0           1               0                 0  Public
1   1            1           0               0                 0  Public
2   2            0           1               0                 0  Public

solution (10000, 6) ['id', 'same_figure', 'same_paper', 'related_papers', 'unrelated_papers', 'Usage']
   id  same_figure  same_paper  related_papers  unrelated_papers   Usage
0   0            1           0               0                 0  Public
1   1            1           0               0                 0  Public
2   2            1           0               0                 0  Public

solution label sums:
 id                  49995000
same_figure             1000
same_paper              3000
related_papers          3000
unrelated_papers        3000
dtype: int64

train cols: ['id',

In [3]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): metadata scan (~420s) — meta_train/test.parquet বানানো হয়ে গেছে
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
import pandas as pd, numpy as np, hashlib, gc, time

COMP = '/kaggle/input/competitions/astroclimb'
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']
PNG = 'iVBORw0KGgo'   # organizer সব image PNG হিসেবে encode করেছে, তাই prefix check exact

def scan(path, has_labels, chunksize=200):
    out, t0, n = [], time.time(), 0
    for ch in pd.read_csv(path, chunksize=chunksize):
        d = pd.DataFrame({'id': ch['id'].values})
        for k, col in (('1','obj_1'), ('2','obj_2')):
            s = ch[col].fillna('')
            d['t'+k]   = np.where(s.str.startswith(PNG), 'image', 'text')
            d['len'+k] = s.str.len().values
            d['h'+k]   = [hashlib.md5(x.encode('utf-8','replace')).hexdigest() for x in s]
        if has_labels:
            d['label'] = ch[LABELS].values.argmax(1)
        out.append(d)
        n += len(ch)
        if n % 2000 == 0:
            print(f'  {n} rows  {time.time()-t0:.0f}s', flush=True)
        del ch; gc.collect()
    return pd.concat(out, ignore_index=True)

print('scanning train...')
mtr = scan(f'{COMP}/train.csv', True)
print('scanning test...')
mte = scan(f'{COMP}/test.csv', False)

mtr['label'] = pd.Categorical.from_codes(mtr.label, LABELS)
mtr.to_parquet('/kaggle/working/meta_train.parquet')
mte.to_parquet('/kaggle/working/meta_test.parquet')
print('done', mtr.shape, mte.shape)


scanning train...
  2000 rows  70s
  4000 rows  92s
  6000 rows  161s
  8000 rows  200s
  10000 rows  218s
scanning test...
  2000 rows  56s
  4000 rows  76s
  6000 rows  137s
  8000 rows  176s
  10000 rows  197s
done (10000, 8) (10000, 7)


In [6]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): solution.csv audit — একবারের EDA, training-এ ব্যবহার নিষেধ
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
sol = pd.read_csv(f'{COMP}/solution.csv')
print(sol.Usage.value_counts())

sol['label'] = sol[LABELS].idxmax(1)
print('\nlabel x Usage:'); print(pd.crosstab(sol.label, sol.Usage))

# key টা কি label অনুযায়ী sorted? (train-এর মতো)
print('\nfirst 20 labels:', sol.label.head(20).tolist())
print('label changes down the file:', int((sol.label != sol.label.shift()).sum()))


Usage
Public    10000
Name: count, dtype: int64

label x Usage:
Usage             Public
label                   
related_papers      3000
same_figure         1000
same_paper          3000
unrelated_papers    3000

first 20 labels: ['same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure', 'same_figure']
label changes down the file: 4


In [7]:
import pandas as pd, numpy as np
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']

mtr = pd.read_parquet(f'{W}/meta_train.parquet')
mte = pd.read_parquet(f'{W}/meta_test.parquet')
combo = lambda d: np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['combo'], mte['combo'] = combo(mtr), combo(mte)

print('TRAIN label counts:'); print(mtr.label.value_counts())
print('\nTRAIN modality:'); print(mtr.combo.value_counts())
print('\nTEST  modality:');  print(mte.combo.value_counts())

print('\n--- modality x label (row %) ---')
print(pd.crosstab(mtr.combo, mtr.label, normalize='index').round(3)*100)
print('\n--- counts ---')
print(pd.crosstab(mtr.combo, mtr.label))

# train-ও কি label-sorted?
print('\ntrain label changes down file:', int((mtr.label != mtr.label.shift()).sum()))


TRAIN label counts:
label
same_paper          3000
related_papers      3000
unrelated_papers    3000
same_figure         1000
Name: count, dtype: int64

TRAIN modality:
combo
image+text     4000
image+image    3000
text+text      3000
Name: count, dtype: int64

TEST  modality:
combo
image+text     4000
image+image    3000
text+text      3000
Name: count, dtype: int64

--- modality x label (row %) ---
label        same_figure  same_paper  related_papers  unrelated_papers
combo                                                                 
image+image          0.0        33.3            33.3              33.3
image+text          25.0        25.0            25.0              25.0
text+text            0.0        33.3            33.3              33.3

--- counts ---
label        same_figure  same_paper  related_papers  unrelated_papers
combo                                                                 
image+image            0        1000            1000              1000
image+text  

In [8]:
# --- object overlap: split strategy এখান থেকেই ঠিক হবে ---
tro = pd.concat([mtr.h1, mtr.h2]); teo = pd.concat([mte.h1, mte.h2])
ov = set(tro) & set(teo)
print('unique objects  train:', tro.nunique(), '| test:', teo.nunique(),
      '| total:', len(set(tro) | set(teo)))
print('train∩test objects:', len(ov), f'({len(ov)/teo.nunique():.1%} of test objects)')
print('\nobject repeat count (train):')
print(tro.value_counts().value_counts().sort_index().head(8))

# --- symmetry ---
ko = mtr.h1 + '|' + mtr.h2
ku = np.where(mtr.h1 < mtr.h2, mtr.h1+'|'+mtr.h2, mtr.h2+'|'+mtr.h1)
print('\nexact duplicate ordered pairs:', int(ko.duplicated().sum()))
print('same pair both directions:', int(pd.Series(ku).duplicated().sum() - ko.duplicated().sum()))
print('unordered pairs w/ conflicting labels:',
      int((mtr.groupby(ku).label.nunique() > 1).sum()))

# --- size proxy ---
print('\nbase64 KB by modality:')
print(pd.concat([mtr[['len1','t1']].rename(columns={'len1':'len','t1':'t'}),
                 mtr[['len2','t2']].rename(columns={'len2':'len','t2':'t'})])
        .groupby('t')['len'].describe(percentiles=[.5,.95,.99]).div(1024).round(1))


unique objects  train: 18666 | test: 10514 | total: 29179
train∩test objects: 1 (0.0% of test objects)

object repeat count (train):
count
1    17401
2     1197
3       67
4        1
Name: count, dtype: int64

exact duplicate ordered pairs: 0
same pair both directions: 0
unordered pairs w/ conflicting labels: 0

base64 KB by modality:
       count   mean     std   min    50%     95%     99%      max
t                                                                
image    9.8  985.5  1216.1  22.8  592.5  3106.7  5737.5  19060.3
text     9.8    0.5     0.3   0.0    0.4     1.1     1.5      3.1


In [9]:
from sklearn.metrics import f1_score
y = mtr.label.astype(str).values

print('all-same_paper      :', round(f1_score(y, np.full(len(y),'same_paper'), average='macro'), 4))

rng = np.random.default_rng(0)
print('uniform random      :', round(f1_score(y, rng.choice(LABELS, len(y)), average='macro'), 4))

# modality constraint মেনে random (same_figure শুধু image+text-এ)
pred = np.where(mtr.combo.values == 'image+text',
                rng.choice(LABELS, len(y)),
                rng.choice(LABELS[1:], len(y)))
print('constraint-aware rnd:', round(f1_score(y, pred, average='macro'), 4))


all-same_paper      : 0.1154
uniform random      : 0.2371
constraint-aware rnd: 0.2896


In [10]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): image+text cache (~1300s) — img384/ আর texts.parquet তৈরি আছে
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
import os, gc, time, base64, hashlib, html, re, unicodedata
import pandas as pd, numpy as np
from io import BytesIO
from PIL import Image
Image.MAX_IMAGE_PIXELS = None          # বড় figure-এ decompression-bomb warning বন্ধ

COMP = '/kaggle/input/competitions/astroclimb'
# OUT / IMGD আসে CELL 0R থেকে
os.makedirs(IMGD, exist_ok=True)
PNG, SIDE = 'iVBORw0KGgo', 384        # 384 রাখছি: 224-এ figure-এর axis label/tick মুছে যায়

_WS = re.compile(r'\s+')
def clean_text(s):
    # conservative: LaTeX, unit, 'SN 2011fe'-এর মতো identifier সব signal — ছুঁচ্ছি না
    s = unicodedata.normalize('NFKC', html.unescape(s))
    return _WS.sub(' ', s).strip()

def obj_hash(s):
    return hashlib.md5(s.encode('utf-8','replace')).hexdigest()

texts, seen, fails = {}, set(), []

def handle(s):
    h = obj_hash(s)
    if h in seen:
        return h
    seen.add(h)
    if s.startswith(PNG):
        try:
            img = Image.open(BytesIO(base64.b64decode(s)))
            img.draft('RGB', (SIDE, SIDE))          # PNG-তে no-op, কিন্তু নিরাপদ
            img = img.convert('RGB')
            img.thumbnail((SIDE, SIDE), Image.LANCZOS)
            img.save(f'{IMGD}/{h}.jpg', 'JPEG', quality=90)
        except Exception as e:
            fails.append((h, repr(e)[:80]))
    else:
        texts[h] = clean_text(s)
    return h

def cache(path, chunksize=200):
    t0 = time.time(); n = 0
    for ch in pd.read_csv(path, chunksize=chunksize):
        for col in ('obj_1', 'obj_2'):
            for s in ch[col].values:
                handle(s)
        n += len(ch)
        if n % 1000 == 0:
            print(f'  {n}  {time.time()-t0:.0f}s  imgs={len(seen)-len(texts)} txt={len(texts)} fail={len(fails)}', flush=True)
        del ch; gc.collect()

print('caching train...'); cache(f'{COMP}/train.csv')
print('caching test...');  cache(f'{COMP}/test.csv')

pd.DataFrame({'hash': list(texts), 'text': list(texts.values())}).to_parquet(f'{OUT}/texts.parquet')
print('\nunique objects:', len(seen), '| texts:', len(texts),
      '| images:', len(os.listdir(IMGD)), '| failures:', len(fails))
print(fails[:5])


caching train...
  1000  77s  imgs=1000 txt=999 fail=0
  2000  244s  imgs=2980 txt=999 fail=0
  3000  321s  imgs=3928 txt=1985 fail=0
  4000  321s  imgs=3928 txt=3931 fail=0
  5000  479s  imgs=5805 txt=3931 fail=0
  6000  561s  imgs=6699 txt=4869 fail=0
  7000  561s  imgs=6699 txt=6717 fail=0
  8000  719s  imgs=8481 txt=6717 fail=0
  9000  793s  imgs=9341 txt=7602 fail=0
  10000  794s  imgs=9341 txt=9325 fail=0
caching test...
  1000  83s  imgs=10341 txt=10321 fail=0
  2000  207s  imgs=11871 txt=10321 fail=0
  3000  259s  imgs=12479 txt=11187 fail=0
  4000  260s  imgs=12479 txt=12503 fail=0
  5000  363s  imgs=13331 txt=12503 fail=0
  6000  401s  imgs=13601 txt=12943 fail=0
  7000  402s  imgs=13601 txt=13603 fail=0
  8000  479s  imgs=14317 txt=13603 fail=0
  9000  514s  imgs=14598 txt=13960 fail=0
  10000  514s  imgs=14598 txt=14581 fail=0

unique objects: 29179 | texts: 14581 | images: 14598 | failures: 0
[]


In [13]:
import torch, numpy as np, pandas as pd, os, time
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPModel, CLIPProcessor, AutoTokenizer, AutoModel, AutoImageProcessor

dev   = 'cuda'
# CACHE / IMGD / OUT সব আসে CELL 0R থেকে

texts = pd.read_parquet(f'{CACHE}/texts.parquet')
img_hashes = sorted(h[:-4] for h in os.listdir(IMGD))
print('texts:', len(texts), '| images:', len(img_hashes))

def norm(x):
    return x / np.linalg.norm(x, axis=1, keepdims=True).clip(1e-8)

def to_tensor(out):
    """encoder-ভেদে output tensor বা ModelOutput — দুইটাই সামলায়"""
    if torch.is_tensor(out):
        return out
    if getattr(out, 'pooler_output', None) is not None:
        return out.pooler_output
    return out.last_hidden_state[:, 0]          # CLS token

class ImgDS(Dataset):
    def __init__(self, hashes, proc): self.h, self.p = hashes, proc
    def __len__(self): return len(self.h)
    def __getitem__(self, i):
        im = Image.open(f'{IMGD}/{self.h[i]}.jpg').convert('RGB')
        return self.p(images=im, return_tensors='pt')['pixel_values'][0]

@torch.no_grad()
def embed_images(proc, fn, bs=48):
    dl = DataLoader(ImgDS(img_hashes, proc), batch_size=bs, num_workers=4, pin_memory=True)
    out, t0 = [], time.time()
    for i, b in enumerate(dl):
        out.append(to_tensor(fn(b.to(dev, torch.float16))).float().cpu().numpy())
        if i % 50 == 0:
            print(f'  {i*bs}/{len(img_hashes)}  {time.time()-t0:.0f}s', flush=True)
    return norm(np.concatenate(out))

@torch.no_grad()
def embed_texts(fn, bs=128):
    out, t0 = [], time.time()
    for i in range(0, len(texts), bs):
        out.append(fn(texts.text.iloc[i:i+bs].tolist()))
        if (i // bs) % 20 == 0:
            print(f'  {i}/{len(texts)}  {time.time()-t0:.0f}s', flush=True)
    return norm(np.concatenate(out))


texts: 14581 | images: 14598


In [14]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): CLIP embedding (~200s) — emb_clip_img/txt.npy সেভ করা আছে
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
MODEL = 'openai/clip-vit-large-patch14'
clip  = CLIPModel.from_pretrained(MODEL, torch_dtype=torch.float16).to(dev).eval()
cproc = CLIPProcessor.from_pretrained(MODEL)

ci = embed_images(cproc.image_processor, clip.get_image_features)
np.save(f'{OUT}/emb_clip_img.npy', ci); print('clip img', ci.shape)

def clip_txt(batch):
    tk = cproc.tokenizer(batch, padding=True, truncation=True,
                         max_length=77, return_tensors='pt').to(dev)   # CLIP-এ মাত্র 77 token
    return to_tensor(clip.get_text_features(**tk)).float().cpu().numpy()

ct = embed_texts(clip_txt)
np.save(f'{OUT}/emb_clip_txt.npy', ct); print('clip txt', ct.shape)

del clip; torch.cuda.empty_cache()


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-large-patch14
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0/14598  2s
  2400/14598  26s
  4800/14598  52s
  7200/14598  82s
  9600/14598  110s
  12000/14598  138s
  14400/14598  166s
clip img (14598, 768)
  0/14581  0s
  2560/14581  4s
  5120/14581  7s
  7680/14581  10s
  10240/14581  13s
  12800/14581  17s
clip txt (14581, 768)


In [15]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): SciNCL embedding — emb_sci_txt.npy সেভ করা আছে
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
tok = AutoTokenizer.from_pretrained('malteos/scincl')
sci = AutoModel.from_pretrained('malteos/scincl', torch_dtype=torch.float16).to(dev).eval()

def sci_txt(batch):
    tk = tok(batch, padding=True, truncation=True, max_length=512, return_tensors='pt').to(dev)
    return to_tensor(sci(**tk)).float().cpu().numpy()

st = embed_texts(sci_txt)
np.save(f'{OUT}/emb_sci_txt.npy', st); print('scincl txt', st.shape)

del sci; torch.cuda.empty_cache()


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/327 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: malteos/scincl
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  0/14581  1s
  2560/14581  19s
  5120/14581  39s
  7680/14581  57s
  10240/14581  75s
  12800/14581  93s
scincl txt (14581, 768)


In [16]:
%%script false --no-raise-error
# ⏭️ SKIPPED (resume mode): DINOv2 embedding — emb_dino_img.npy সেভ করা আছে
# আবার চালাতে হলে উপরের দুই লাইন মুছে দাও।
dproc = AutoImageProcessor.from_pretrained('facebook/dinov2-base')
dino  = AutoModel.from_pretrained('facebook/dinov2-base', torch_dtype=torch.float16).to(dev).eval()

di = embed_images(dproc, lambda x: dino(pixel_values=x))
np.save(f'{OUT}/emb_dino_img.npy', di); print('dino img', di.shape)

del dino; torch.cuda.empty_cache()

# row order ↔ hash mapping — এটা ছাড়া embedding অর্থহীন
pd.DataFrame({'hash': img_hashes}).to_parquet(f'{OUT}/img_hashes.parquet')
texts[['hash']].to_parquet(f'{OUT}/txt_hashes.parquet')
print('saved: clip_img, clip_txt, sci_txt, dino_img + hash maps')


preprocessor_config.json:   0%|          | 0.00/436 [00:00<?, ?B/s]

The image processor of type `BitImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json:   0%|          | 0.00/548 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

  0/14598  2s
  2400/14598  10s
  4800/14598  20s
  7200/14598  29s
  9600/14598  39s
  12000/14598  49s
  14400/14598  58s
dino img (14598, 768)
saved: clip_img, clip_txt, sci_txt, dino_img + hash maps


In [17]:
import os, glob
print(glob.glob(f'{W}/*.npy')); print(glob.glob(f'{W}/meta_*.parquet'))


['/kaggle/working/meta_train.parquet', '/kaggle/working/meta_test.parquet']
[]


In [18]:
import numpy as np, pandas as pd

# W আসে CELL 0R থেকে
META = f'{W}/meta_train.parquet'        # attach করলে /kaggle/input/<slug>/meta_train.parquet

m = pd.read_parquet(META)
m['combo'] = np.where(m.t1 < m.t2, m.t1+'+'+m.t2, m.t2+'+'+m.t1)

img_h = pd.read_parquet(f'{W}/img_hashes.parquet').hash.values
txt_h = pd.read_parquet(f'{W}/txt_hashes.parquet').hash.values
img_ix = {h: i for i, h in enumerate(img_h)}
txt_ix = {h: i for i, h in enumerate(txt_h)}

E = {k: np.load(f'{W}/emb_{k}.npy') for k in
     ['clip_img', 'clip_txt', 'sci_txt', 'dino_img']}

def lookup(hashes, types, space):
    """space: 'clip' (দুই modality একসাথে), 'sci' (text), 'dino' (image)"""
    out = np.zeros((len(hashes), 768), dtype=np.float32)
    ok  = np.zeros(len(hashes), dtype=bool)
    for i, (h, t) in enumerate(zip(hashes, types)):
        if t == 'image' and space in ('clip', 'dino') and h in img_ix:
            out[i] = E['clip_img' if space == 'clip' else 'dino_img'][img_ix[h]]; ok[i] = True
        elif t == 'text' and space in ('clip', 'sci') and h in txt_ix:
            out[i] = E['clip_txt' if space == 'clip' else 'sci_txt'][txt_ix[h]]; ok[i] = True
    return out, ok

for space in ['clip', 'sci', 'dino']:
    e1, ok1 = lookup(m.h1.values, m.t1.values, space)
    e2, ok2 = lookup(m.h2.values, m.t2.values, space)
    m[f'cos_{space}'] = np.where(ok1 & ok2, (e1 * e2).sum(1), np.nan)

print('coverage (কত row-তে দুই দিকেই embedding পাওয়া গেল):')
print(m[['cos_clip','cos_sci','cos_dino']].notna().sum())

print('\n--- mean cosine by label x combo ---')
for space in ['clip', 'sci', 'dino']:
    t = m.pivot_table(index='combo', columns='label', values=f'cos_{space}', aggfunc='mean')
    if t.notna().any().any():
        print(f'\n[{space}]'); print(t.round(3))


coverage (কত row-তে দুই দিকেই embedding পাওয়া গেল):
cos_clip    10000
cos_sci      3000
cos_dino     3000
dtype: int64

--- mean cosine by label x combo ---

[clip]
label        same_figure  same_paper  related_papers  unrelated_papers
combo                                                                 
image+image          NaN       0.769           0.723             0.702
image+text          0.26       0.231           0.222             0.204
text+text            NaN       0.537           0.469             0.420

[sci]
label      same_paper  related_papers  unrelated_papers
combo                                                  
text+text       0.925           0.908             0.883

[dino]
label        same_paper  related_papers  unrelated_papers
combo                                                    
image+image        0.58           0.528             0.486


/tmp/ipykernel_58/2177988501.py:38: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  t = m.pivot_table(index='combo', columns='label', values=f'cos_{space}', aggfunc='mean')
/tmp/ipykernel_58/2177988501.py:38: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  t = m.pivot_table(index='combo', columns='label', values=f'cos_{space}', aggfunc='mean')
/tmp/ipykernel_58/2177988501.py:38: FutureWarning: The default value of observed=False is deprecated and will change to observed=True in a future version of pandas. Specify observed=False to silence this warning and retain the current behavior
  t = m.pivot_table(index='combo', columns='label', values=f'cos_{space}', aggfunc='me

In [19]:
import numpy as np, pandas as pd, os
# W আসে CELL 0R থেকে
LABELS = ['same_figure','same_paper','related_papers','unrelated_papers']

mtr = pd.read_parquet(f'{W}/meta_train.parquet')   # attach করলে path বদলাও
mte = pd.read_parquet(f'{W}/meta_test.parquet')
for d in (mtr, mte):
    d['combo'] = np.where(d.t1 < d.t2, d.t1+'+'+d.t2, d.t2+'+'+d.t1)
mtr['y'] = mtr.label.astype(str)

img_ix = {h: i for i, h in enumerate(pd.read_parquet(f'{W}/img_hashes.parquet').hash.values)}
txt_ix = {h: i for i, h in enumerate(pd.read_parquet(f'{W}/txt_hashes.parquet').hash.values)}
E = {k: np.load(f'{W}/emb_{k}.npy') for k in ['clip_img','clip_txt','sci_txt','dino_img']}

def center(x):
    """anisotropy কমায়: গড় সরিয়ে আবার normalize"""
    z = x - x.mean(0, keepdims=True)
    return z / np.linalg.norm(z, axis=1, keepdims=True).clip(1e-8)

E['sci_txt_c']  = center(E['sci_txt'])
E['dino_img_c'] = center(E['dino_img'])

def side(hashes, types, space):
    """space: clip | sci | dino — দুই পাশের embedding matrix"""
    out = np.zeros((len(hashes), 768), dtype=np.float32)
    for i, (h, t) in enumerate(zip(hashes, types)):
        if t == 'image':
            if space == 'clip':  out[i] = E['clip_img'][img_ix[h]]
            elif space == 'dino': out[i] = E['dino_img_c'][img_ix[h]]
        else:
            if space == 'clip':  out[i] = E['clip_txt'][txt_ix[h]]
            elif space == 'sci':  out[i] = E['sci_txt_c'][txt_ix[h]]
    return out

def pair_block(e1, e2, tag):
    d = np.abs(e1 - e2); p = e1 * e2
    cos = (e1 * e2).sum(1); l2 = np.linalg.norm(e1 - e2, axis=1)
    X = np.hstack([d, p, cos[:, None], l2[:, None]])
    cols = ([f'{tag}_d{i}' for i in range(e1.shape[1])] +
            [f'{tag}_p{i}' for i in range(e1.shape[1])] + [f'{tag}_cos', f'{tag}_l2'])
    return X, cols

def build(m, combo):
    """এক subset-এর feature matrix। image+text-এ শুধু CLIP, বাকিতে CLIP + domain encoder।"""
    d = m[m.combo == combo].reset_index(drop=True)
    spaces = {'image+text': ['clip'], 'text+text': ['clip','sci'],
              'image+image': ['clip','dino']}[combo]
    blocks, cols = [], []
    for sp in spaces:
        e1, e2 = side(d.h1.values, d.t1.values, sp), side(d.h2.values, d.t2.values, sp)
        X, c = pair_block(e1, e2, sp); blocks.append(X); cols += c
    # meta: symmetric রাখতে min/max ব্যবহার করছি, len1/len2 নয়
    mn = np.minimum(d.len1, d.len2).values; mx = np.maximum(d.len1, d.len2).values
    blocks.append(np.c_[mn, mx, mx / np.maximum(mn, 1)])
    cols += ['len_min','len_max','len_ratio']
    return d, np.hstack(blocks).astype(np.float32), cols

for c in ['image+text','text+text','image+image']:
    d, X, cols = build(mtr, c)
    print(c, X.shape, '| classes:', d.y.nunique())


image+text (4000, 1541) | classes: 4
text+text (3000, 3079) | classes: 3
image+image (3000, 3079) | classes: 3


In [20]:
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

SWAP = True          # symmetry augmentation on/off — ablation-এর জন্য
oof_all, y_all, models = [], [], {}

for combo in ['image+text','text+text','image+image']:
    d, X, cols = build(mtr, combo)
    classes = sorted(d.y.unique())                      # 3 বা 4, subset অনুযায়ী
    y = d.y.map({c: i for i, c in enumerate(classes)}).values
    oof = np.zeros((len(y), len(classes)))

    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        Xtr, ytr = X[tr_i], y[tr_i]
        if SWAP:   # |d| আর p symmetric, তাই swap-এ ওগুলো অপরিবর্তিত; cos/l2/meta-ও তাই
            Xtr, ytr = np.vstack([Xtr, Xtr]), np.concatenate([ytr, ytr])
        clf = lgb.LGBMClassifier(objective='multiclass', num_class=len(classes),
                                 n_estimators=600, learning_rate=0.05,
                                 num_leaves=31, colsample_bytree=0.3,
                                 subsample=0.8, subsample_freq=1,
                                 class_weight='balanced', verbose=-1, n_jobs=-1)
        clf.fit(Xtr, ytr)
        oof[va_i] = clf.predict_proba(X[va_i])

    pred = [classes[i] for i in oof.argmax(1)]
    print(f'{combo:12s} macro-F1 (subset): {f1_score(d.y, pred, average="macro"):.4f}')
    oof_all += pred; y_all += d.y.tolist()
    models[combo] = classes

print('\n=== OVERALL macro-F1:', round(f1_score(y_all, oof_all, average='macro'), 4),
      ' (floor 0.2896) ===')


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


image+text   macro-F1 (subset): 0.3964


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


text+text    macro-F1 (subset): 0.5529


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


image+image  macro-F1 (subset): 0.4548

=== OVERALL macro-F1: 0.4806  (floor 0.2896) ===


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [21]:
import numpy as np, pandas as pd, lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

sub = pd.DataFrame({'id': mte.id.values})
for c in LABELS: sub[c] = 0
oof_store = {}

for combo in ['image+text','text+text','image+image']:
    d,  X,  cols = build(mtr, combo)
    dt, Xt, _    = build(mte, combo)
    classes = sorted(d.y.unique())
    y = d.y.map({c: i for i, c in enumerate(classes)}).values

    oof  = np.zeros((len(y), len(classes)))
    test = np.zeros((len(dt), len(classes)))

    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        Xtr, ytr = np.vstack([X[tr_i]]*2), np.concatenate([y[tr_i]]*2)   # swap aug
        clf = lgb.LGBMClassifier(objective='multiclass', num_class=len(classes),
                                 n_estimators=600, learning_rate=0.05, num_leaves=31,
                                 colsample_bytree=0.3, subsample=0.8, subsample_freq=1,
                                 class_weight='balanced', verbose=-1, n_jobs=-1)
        clf.fit(Xtr, ytr)
        oof[va_i] = clf.predict_proba(X[va_i])
        test += clf.predict_proba(Xt) / 5

    oof_store[combo] = (d, oof, classes)
    for j, c in enumerate(classes):                     # test rows-এ বসাও
        sub.loc[sub.id.isin(dt.id), c] = 0
    hard = np.zeros((len(dt), 4), dtype=int)
    for j, c in enumerate(classes):
        hard[:, LABELS.index(c)] = 0
    pick = test.argmax(1)
    for i, p in enumerate(pick):
        sub.loc[sub.id == dt.id.iloc[i], classes[p]] = 1
    print(combo, 'done')

sub[['id'] + LABELS].to_csv(f'{OUT}/submission.csv', index=False)
print(sub[LABELS].sum())          # প্রতিটা class-এ কতগুলো — uniform-এর কাছাকাছি হওয়া উচিত
print(sub.shape, sub[LABELS].sum(1).value_counts())   # প্রতিটা row-তে ঠিক একটা 1


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

image+text done


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

text+text done


/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/ut

image+image done
same_figure         1276
same_paper          2442
related_papers      2239
unrelated_papers    4043
dtype: int64
(10000, 5) 1    10000
Name: count, dtype: int64


In [28]:
for combo, (d, oof, classes) in oof_store.items():
    pred = pd.Series([classes[i] for i in oof.argmax(1)])
    print(combo); print(pred.value_counts().to_dict(), '\n')


image+text
{'same_figure': 1231, 'unrelated_papers': 1225, 'related_papers': 820, 'same_paper': 724} 

text+text
{'unrelated_papers': 1330, 'same_paper': 1007, 'related_papers': 663} 

image+image
{'unrelated_papers': 1250, 'same_paper': 946, 'related_papers': 804} 



In [29]:
from sklearn.metrics import classification_report, confusion_matrix
for combo, (d, oof, classes) in oof_store.items():
    pred = [classes[i] for i in oof.argmax(1)]
    print('='*60); print(combo)
    print(classification_report(d.y, pred, digits=3))
    print(pd.DataFrame(confusion_matrix(d.y, pred, labels=classes),
                       index=classes, columns=classes))


image+text
                  precision    recall  f1-score   support

  related_papers      0.316     0.259     0.285      1000
     same_figure      0.510     0.628     0.563      1000
      same_paper      0.305     0.221     0.256      1000
unrelated_papers      0.438     0.536     0.482      1000

        accuracy                          0.411      4000
       macro avg      0.392     0.411     0.396      4000
    weighted avg      0.392     0.411     0.396      4000

                  related_papers  same_figure  same_paper  unrelated_papers
related_papers               259          213         188               340
same_figure                  133          628         156                83
same_paper                   231          282         221               266
unrelated_papers             197          108         159               536
text+text
                  precision    recall  f1-score   support

  related_papers      0.449     0.298     0.358      1000
      same_pape

In [30]:
%%script false --no-raise-error
# ⏭️ SKIPPED: এই cell GPU চায় (CLIP reload)। CPU background run-এ error দিয়ে
#    পুরো commit ব্যর্থ করে দিত। chunked CLIP লাগলে আলাদাভাবে GPU on করে
#    উপরের দুই লাইন মুছে শুধু 0R → 9 → এই cell চালাও।
# ===== NOTEBOOK A — chunked CLIP text (cell 9 skip করা আছে, তাই CLIP আবার load) =====
import torch, numpy as np
from transformers import CLIPModel, CLIPProcessor
dev, OUT = 'cuda', '/kaggle/working'
MODEL = 'openai/clip-vit-large-patch14'
clip  = CLIPModel.from_pretrained(MODEL, torch_dtype=torch.float16).to(dev).eval()
cproc = CLIPProcessor.from_pretrained(MODEL)

@torch.no_grad()
def clip_txt_chunked(batch, stride=60):
    """caption 77 token ছাড়ালে টুকরো করে গড় — লেজটা আর কাটা যায় না"""
    outs = []
    for s in batch:
        ids = cproc.tokenizer(s, add_special_tokens=False)['input_ids']
        chunks = [ids[i:i+75] for i in range(0, max(len(ids), 1), stride)][:6] or [[]]
        tk = cproc.tokenizer([cproc.tokenizer.decode(c) for c in chunks],
                             padding=True, truncation=True, max_length=77,
                             return_tensors='pt').to(dev)
        outs.append(clip.get_text_features(**tk).float().mean(0).cpu().numpy())
    return np.stack(outs)

ctc = embed_texts(clip_txt_chunked, bs=64)          # embed_texts আসে cell 8 থেকে
np.save(f'{OUT}/emb_clip_txt_chunk.npy', ctc)
print('clip txt chunked', ctc.shape)                # emb_clip_txt.npy মুছবে না — ablation লাগবে

del clip; torch.cuda.empty_cache()


In [1]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score

for combo in ['image+text','text+text','image+image']:
    d, X, cols = build(mtr, combo)
    classes = sorted(d.y.unique())
    y = d.y.map({c: i for i, c in enumerate(classes)}).values
    oof_m = np.zeros((len(y), len(classes)))

    for tr_i, va_i in StratifiedKFold(5, shuffle=True, random_state=0).split(X, y):
        mlp = make_pipeline(
            StandardScaler(),
            MLPClassifier(hidden_layer_sizes=(256,), alpha=1e-3, max_iter=300,
                          early_stopping=True, n_iter_no_change=15, random_state=0))
        mlp.fit(X[tr_i], y[tr_i])
        oof_m[va_i] = mlp.predict_proba(X[va_i])

    lgb_oof = oof_store[combo][1]
    for w in [0.0, 0.3, 0.5, 0.7, 1.0]:
        blend = (1-w) * lgb_oof + w * oof_m
        s = f1_score(d.y, [classes[i] for i in blend.argmax(1)], average='macro')
        print(f'{combo:12s} w_mlp={w:.1f}  {s:.4f}')
    print()


NameError: name 'build' is not defined